In [ ]:
!pip install transformers torch sentencepiece --quiet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
MODEL_NAME = "google/flan-t5-base"

print(f"Loading tokenizer for '{MODEL_NAME}'...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model '{MODEL_NAME}'...")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.eval()

print("\nModel and tokenizer loaded successfully!")

In [ ]:
def generate_response(user_input, conversation_history):
    """
    Generate a factual chatbot response using Flan-T5.

    Parameters:
        user_input (str)            : The latest message from the user.
        conversation_history (list) : List of past (user, bot) turn pairs for context.

    Returns:
        response (str)              : The generated chatbot reply.
        conversation_history (list) : Updated conversation history.
    """

    context = ""
    for past_user, past_bot in conversation_history[-3:]:
        context += f"User: {past_user}\nAssistant: {past_bot}\n"

    prompt = (
        "You are a helpful and knowledgeable AI assistant. "
        "Answer the user's question clearly and accurately.\n\n"
        f"{context}"
        f"User: {user_input}\n"
        "Assistant:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3
        )


    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)


    conversation_history.append((user_input, response))

    return response, conversation_history


print("Response generation function defined successfully!")

Response generation function defined successfully!


In [ ]:
def run_chatbot():
    """
    Main chatbot loop.
    - Continuously accepts user input from the console.
    - Generates and displays factual responses using Flan-T5.
    - Maintains conversation history for context-aware replies.
    - Terminates when user types 'exit' or 'quit'.
    """

    print("=" * 55)
    print("   Chatbot using Hugging Face Transformers (Flan-T5)")
    print("=" * 55)
    print("Type 'exit' or 'quit' to end the conversation.\n")
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    conversation_history = []
    while True:
        user_input = input("You: ").strip()

        if not user_input:
            print("Chatbot: Please type something so I can respond!\n")
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Goodbye! It was great talking to you. Have a wonderful day!")
            print("=" * 55)
            break
        response, conversation_history = generate_response(user_input, conversation_history)

        # Display the chatbot's response
        print(f"Chatbot: {response}\n")


# Start the chatbot
run_chatbot()

   Chatbot using Hugging Face Transformers (Flan-T5)
Type 'exit' or 'quit' to end the conversation.

Chatbot: Hello! I am your AI assistant. How can I help you today?



##Sample Interaction Output

Below is a sample of what a typical chatbot session looks like:

```
=======================================================
   Chatbot using Hugging Face Transformers (Flan-T5)
=======================================================
Type 'exit' or 'quit' to end the conversation.

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: Hello
Chatbot: Hello! How can I assist you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence is the simulation of human intelligence
         in machines that are programmed to think, learn, and solve problems.

You: Who created Python?
Chatbot: Python was created by Guido van Rossum and first released in 1991.

You: Thank you
Chatbot: You're welcome! Feel free to ask more questions.

You: exit
Chatbot: Goodbye! It was great talking to you. Have a wonderful day!
=======================================================
```